# Kamon Dataset Exploration with Pixeltable
This notebook demonstrates how to explore the Kamon dataset.

In [1]:
import pixeltable as pxt

In [ ]:
# First, create a local table from the published source (only need to do this once):
kamon = pxt.create_table(
     'kamon_db.kamon_images',
     source='pxt://pixeltable:demos/kamon/kamon_images'
)

In [2]:
# List available tables
pxt.list_tables()

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/alison-pxt/.pixeltable/pgdata


['kamon_db.images',
 'kamon_db.translations',
 'kamon_db.parsed_raw',
 'kamon_db.kamon_images']

In [ ]:
# Get the main kamon table
# Uncomment if you disconnect the Pixeltable from a variable
# Or if you have rebuilt the tables from source via the scripts
# kamon = pxt.get_table('kamon_db.kamon_images')

In [ ]:
# Check how many images we have
kamon.count()

1002

In [ ]:
# See the table schema
kamon.describe()

In [ ]:
# View first rows
kamon.head()

In [ ]:
# Check that translations are joined
kamon.where(kamon.translation.contains('moon')).collect()

## Where are the images stored?

**Short answer**: In Pixeltable's cache at `~/.pixeltable/file_cache/`

**How it works**:
1. When you created the table, images were loaded from GitHub URLs
2. Pixeltable automatically downloaded and cached each image locally
3. The `image` column stores a reference to the cached file
4. You can access both the original URL and the cached location

Let's see both:


In [ ]:
# See where images are stored
kamon.select(
    kamon.image,
    kamon.image.fileurl,    # Original GitHub URL
    kamon.image.localpath   # Where Pixeltable cached it locally
).head(3)

**Key points:**
- `fileurl` - The original source (shareable URL)
- `localpath` - Where Pixeltable cached it on your machine
- These are computed on-demand, not stored as columns
- Images work offline after initial download
- When you share the table, others download from the same GitHub URLs

This makes the table portable - it works on any machine.


## Understanding the embedding indexes

This dataset includes **two precomputed embedding indexes** that enable semantic search:

1. `clip_idx` - Image Embeddings (Multimodal)
- **Indexed on**: The `image` column
- **Model**: CLIP (openai/clip-vit-base-patch32)
- **What it does**: Converts each image into a vector embedding
- **Why multimodal**: CLIP understands BOTH images and text in the same embedding space
- **You can search with**:
  - Text query → finds images matching that description
  - Image query → finds visually similar images

2. `text_idx` - Translation Embeddings
- **Indexed on**: The `translation` column (English text)
- **Model**: Sentence Transformer (all-MiniLM-L6-v2)
- **What it does**: Converts each translation into a vector embedding
- **You can search with**:
  - Text query → finds translations with similar meaning

---

## Search images with CLIP

CLIP lets you find images using text descriptions or other images as queries.


In [ ]:
# Example: Use text "moon and stars" to find images with those visual elements
# Note: kamon.image.similarity() because we're searching the image column
sim = kamon.image.similarity('moon and stars', idx='clip_idx')
kamon.order_by(sim, asc=False).select(
    kamon.image,
    kamon.translation,
    kamon.description,
    sim  # ← Add this to show the similarity score
).limit(5).collect()

### Keyword vs CLIP search

Compare keyword search (exact text match) with CLIP semantic search:


In [ ]:
# Keyword search: Searches translation TEXT for exact word "moon"
kamon.where(kamon.translation.contains('moon')).collect()


In [ ]:
# CLIP search: Uses text to find IMAGES that match the concept visually
sim = kamon.image.similarity('moon', idx='clip_idx')
kamon.order_by(sim, asc=False).select(
    kamon.image,
    kamon.translation,
    sim  # ← Add this to show the similarity score
).limit(5).collect()

### Image-to-image search

Find visually similar kamon using an image as the query:


In [ ]:
# Get a specific image to use as our query (chrysanthemum crane kamon)
sample_row = kamon.where(kamon.github_url.contains('Kiku_Tsuru_inverted.png')).select(kamon.image).collect()[0]
sample_img = sample_row['image']

# Display the query image
sample_img

In [ ]:
# Now find kamon that look visually similar to this chrysanthemum crane
sim = kamon.image.similarity(sample_img, idx='clip_idx')
kamon.order_by(sim, asc=False).select(
    kamon.image,
    kamon.translation,
    sim  # ← Add this to show the similarity score
).limit(5).collect()

## Search translations with text embeddings

This is DIFFERENT from CLIP - here we're finding similar TRANSLATION TEXT, not similar images.

Use the `text_idx` to find translations that are semantically similar to your query:


In [ ]:
# Note: kamon.translation.similarity() because we're searching the translation text
# This finds rows where the translation TEXT is semantically similar to our query
sim = kamon.translation.similarity('chrysanthemum', idx='text_idx')
kamon.order_by(sim, asc=False).select(
    kamon.image,
    kamon.translation,
    sim  # ← Add this to show the similarity score
).limit(5).collect()

## Creating custom views

Views are filtered versions of tables. Let's create views for different image sources:


In [ ]:
# Create views for wiki and edo sources
wiki = pxt.create_view(
    'kamon_db.wiki_images', 
    kamon.where(kamon.source == 'wiki'),
    if_exists='replace')
edo = pxt.create_view(
    'kamon_db.edo_images', 
    kamon.where(kamon.source == 'edo'),
    if_exists='replace')

print(f"Wiki images: {wiki.count()}")
print(f"Edo images: {edo.count()}")

In [ ]:
# Query views like tables
wiki.select(wiki.image, wiki.translation).head(3)

## Combining filters with CLIP search

Filter by source and use CLIP search together:


In [ ]:
# CLIP search within wiki images only
sim = kamon.image.similarity('chrysanthemum', idx='clip_idx')
kamon.where(kamon.source == 'wiki').order_by(sim, asc=False).select(
    kamon.image,
    kamon.translation,
    kamon.description,
    sim  # ← Add this to show the similarity score
).limit(3).collect()

In [ ]:
# Compare: Same search but within edo images
sim = kamon.image.similarity('chrysanthemum', idx='clip_idx')
kamon.where(kamon.source == 'edo').order_by(sim, asc=False).select(
    kamon.image,
    kamon.translation,
    kamon.description,
    sim  # ← Add this to show the similarity score
).limit(3).collect()